ts2 is the cleaned srips_small table. The notebook explain how to get to it. if you run all the notebook you will get ts2 at the end

In [9]:
import pandas as pd

In [10]:
ts = pd.read_csv("../trips_small.csv")
ts

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,battery_end,duration,distance
0,11462,7,1.669377e+09,8.253932,50.069332,8.253393,50.070504,126084480.0,NaN,NaN,NaN,360.0,135.879806
1,11462,7,1.669897e+09,8.242628,50.071337,8.216180,50.095700,87796472.0,4775153.0,NaN,NaN,1920.0,3301.514764
2,11462,7,1.670252e+09,8.216180,50.095700,8.217791,50.095744,4775153.0,NaN,NaN,NaN,780.0,115.020620
3,11462,7,1.670349e+09,8.226448,50.076546,8.249343,50.078160,4775087.0,87796883.0,NaN,NaN,540.0,1643.609250
4,11462,7,1.671514e+09,8.249343,50.078160,8.242628,50.071337,87796883.0,87796472.0,NaN,NaN,4680.0,897.351993
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161371,840054,679,1.684753e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,96.0,1320.0,3061.891906
1161372,840054,679,1.685088e+09,-3.166644,51.446057,-3.163247,51.465538,6270602.0,88102363.0,100.0,97.0,780.0,2178.941052
1161373,840054,679,1.686059e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,95.0,900.0,3061.891906
1161374,840054,679,1.686121e+09,-3.185756,51.429879,-3.163247,51.465538,86312067.0,88102363.0,87.0,75.0,1020.0,4260.899309


In [11]:
ts.isna().mean() * 100

bike_id              0.000000
city_id              0.000000
time_start           0.000000
lon_start            0.000000
lat_start            0.000000
lon_end              0.000000
lat_end              0.000000
station_id_start     7.681836
station_id_end      10.496514
battery_start       85.488593
battery_end         85.479207
duration             0.000000
distance             0.000000
dtype: float64

On a 4 colonnes avec des valeurs qui manquent:
* station_id_start avec 7.68%
* station_is_end avec 10.49%
* battery_start avec 85.48%
* battery_end avec 85.47%

Première chose à faire est de vérifier que les valeurs de batteries qui manquent sont associées à des vélos éléctriques ou classiques.

In [12]:
ts.drop(columns=["bike_id","city_id","time_start","station_id_start","station_id_end"]).describe()

,lon_start,lat_start,lon_end,lat_end,battery_start,battery_end,duration,distance
count,1.161376e+06,1.161376e+06,1.161376e+06,1.161376e+06,168532.000000,168641.000000,1.161376e+06,1.161376e+06
mean,1.143184e+01,4.941432e+01,1.143162e+01,4.941407e+01,80.988697,76.688996,1.256241e+03,1.791774e+03
std,7.320269e+00,3.707710e+00,7.320443e+00,3.708131e+00,21.384192,22.615122,1.980838e+03,1.495191e+03
min,-1.560567e+01,2.774058e+01,-1.560567e+01,2.774058e+01,0.000000,0.000000,1.210000e+02,5.001365e+01
25%,7.165686e+00,4.717771e+01,7.165686e+00,4.717771e+01,66.000000,61.000000,4.800000e+02,8.851685e+02
50%,1.019364e+01,5.027373e+01,1.018726e+01,5.026997e+01,89.000000,83.000000,7.200000e+02,1.346711e+03
75%,1.745811e+01,5.174006e+01,1.745811e+01,5.173991e+01,100.000000,97.000000,1.200000e+03,2.238540e+03
max,3.398531e+01,5.708158e+01,3.398632e+01,5.708158e+01,100.000000,100.000000,2.874000e+04,1.999946e+04


In [13]:
bt = pd.read_csv("../bike_types.csv")

In [14]:
b = pd.read_csv("../bikes.csv")

In [15]:
bikes_info = b.merge(bt,how="left",left_on="bike_type_id",right_on="id",suffixes=["_bike","_bike_type"])
bikes_info.isna().mean() * 100
columns = ["computer_id","id_bike_type","vehicle_image","name","description","form_factor","rider_capacity","max_range","battery_capacity"]
bikes_info.drop(columns=columns)

,id_bike,bike_type_id,propulsion_type
0,400481,196,human
1,374044,204,electric_assist
2,373778,204,electric_assist
3,34749,14,human
4,88359,143,electric_assist
...,...,...,...
88431,372994,204,electric_assist
88432,372649,204,electric_assist
88433,372951,204,electric_assist
88434,374346,204,electric_assist


In [16]:
new_ts = ts.merge(bikes_info,how="left",left_on="bike_id",right_on="id_bike")
new_ts["propulsion_type"].value_counts(normalize=True) * 100

propulsion_type
human              84.461921
electric_assist    15.538079
Name: proportion, dtype: float64

* We can see that 84% of values in propulsion_type is human which is almost equal to 85% missing values of battery status
* We need to check if human propulsion values correspond to the missing values of the battery status

In [17]:
import numpy as np

* On va convertir les valeur "human" dans la colonne propulsion_type en NaN

In [18]:
new_ts["propulsion_type"] = new_ts["propulsion_type"].map({"human": np.nan, "electric_assist": 1})
new_ts.isna().mean() * 100

bike_id              0.000000
city_id              0.000000
time_start           0.000000
lon_start            0.000000
lat_start            0.000000
lon_end              0.000000
lat_end              0.000000
station_id_start     7.681836
station_id_end      10.496514
battery_start       85.488593
battery_end         85.479207
duration             0.000000
distance             0.000000
id_bike              0.000000
bike_type_id         0.000000
computer_id          0.000000
id_bike_type         0.000000
vehicle_image        0.202002
name                 0.202002
description         48.552923
form_factor          0.202002
rider_capacity       0.202002
propulsion_type     84.493308
max_range           84.501488
battery_capacity    84.493308
dtype: float64

* Combien de lignes avec seulement une valeur manquante pour le couple (battery_start, battery_end)

In [19]:
one = new_ts[new_ts["battery_start"].notna() ^ new_ts["battery_end"].notna()]
one

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,...,computer_id,id_bike_type,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity
201998,42622,676,1.670701e+09,7.219444,51.144028,7.122056,51.058917,30240427.0,32135341.0,NaN,...,7551060064,143,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 RFID,- e-eco B frame with Framelock\r\n- aluminum b...,bicycle,1.0,1.0,60000.0,500.0
202097,42623,676,1.684534e+09,7.197056,50.929000,7.196167,50.914278,49442475.0,32136150.0,NaN,...,7551059755,143,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 RFID,- e-eco B frame with Framelock\r\n- aluminum b...,bicycle,1.0,1.0,60000.0,500.0
202098,42623,676,1.684598e+09,7.196167,50.914278,7.182677,50.900797,32136150.0,30240394.0,NaN,...,7551059755,143,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 RFID,- e-eco B frame with Framelock\r\n- aluminum b...,bicycle,1.0,1.0,60000.0,500.0
202101,42623,676,1.684673e+09,7.182677,50.900797,7.176250,50.924389,30240394.0,32136249.0,NaN,...,7551059755,143,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 RFID,- e-eco B frame with Framelock\r\n- aluminum b...,bicycle,1.0,1.0,60000.0,500.0
202104,42623,676,1.685276e+09,7.184889,50.959278,7.161333,50.963972,32134268.0,30239595.0,NaN,...,7551059755,143,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 RFID,- e-eco B frame with Framelock\r\n- aluminum b...,bicycle,1.0,1.0,60000.0,500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1158625,826043,679,1.684855e+09,-3.174022,51.438833,-3.179619,51.477280,30005988.0,60694214.0,NaN,...,7551094076,182,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0
1158645,826044,679,1.663164e+09,-3.206055,51.403857,-3.179534,51.484449,58569525.0,52962132.0,NaN,...,7551092436,182,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0
1158936,825564,679,1.685463e+09,-3.181851,51.448645,-3.166644,51.446057,53040980.0,6270602.0,NaN,...,7551059021,182,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0
1159834,825613,679,1.673042e+09,-3.166644,51.446057,-3.163247,51.465538,6270602.0,88102363.0,NaN,...,7551063848,182,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0


* These battery status values are missing completely at random (MCAR) and it represents 0.01% of table rows, so we decide too delete them

In [20]:
partial = new_ts["battery_start"].isna() ^ new_ts["battery_end"].isna()
new_ts = new_ts[~partial]
new_ts.isna().mean() * 100

bike_id              0.000000
city_id              0.000000
time_start           0.000000
lon_start            0.000000
lat_start            0.000000
lon_end              0.000000
lat_end              0.000000
station_id_start     7.681967
station_id_end      10.496978
battery_start       85.488086
battery_end         85.488086
duration             0.000000
distance             0.000000
id_bike              0.000000
bike_type_id         0.000000
computer_id          0.000000
id_bike_type         0.000000
vehicle_image        0.202026
name                 0.202026
description         48.558565
form_factor          0.202026
rider_capacity       0.202026
propulsion_type     84.503276
max_range           84.511457
battery_capacity    84.503276
dtype: float64

In [21]:
new_ts.isna().mean() * 100

bike_id              0.000000
city_id              0.000000
time_start           0.000000
lon_start            0.000000
lat_start            0.000000
lon_end              0.000000
lat_end              0.000000
station_id_start     7.681967
station_id_end      10.496978
battery_start       85.488086
battery_end         85.488086
duration             0.000000
distance             0.000000
id_bike              0.000000
bike_type_id         0.000000
computer_id          0.000000
id_bike_type         0.000000
vehicle_image        0.202026
name                 0.202026
description         48.558565
form_factor          0.202026
rider_capacity       0.202026
propulsion_type     84.503276
max_range           84.511457
battery_capacity    84.503276
dtype: float64

* Now that we are sure that the missing values correspond to human propulsion we will replace missing values by 0 which is coherent
* The 1% more values that are missing in battery status will be considered as MCAR because we don't see any logical reason why they are missing.

In [22]:
new_ts[["battery_start", "battery_end"]] = new_ts[["battery_start", "battery_end"]].fillna(0)
new_ts.isna().mean() * 100

bike_id              0.000000
city_id              0.000000
time_start           0.000000
lon_start            0.000000
lat_start            0.000000
lon_end              0.000000
lat_end              0.000000
station_id_start     7.681967
station_id_end      10.496978
battery_start        0.000000
battery_end          0.000000
duration             0.000000
distance             0.000000
id_bike              0.000000
bike_type_id         0.000000
computer_id          0.000000
id_bike_type         0.000000
vehicle_image        0.202026
name                 0.202026
description         48.558565
form_factor          0.202026
rider_capacity       0.202026
propulsion_type     84.503276
max_range           84.511457
battery_capacity    84.503276
dtype: float64

* For the station_ids that are missing we need to check whether the city allows the bikes to be returned away from stations

In [23]:
c = pd.read_csv("../cities.csv")
c

,id,name,lat,lon,timezone,country,return_to_official_only
0,220,Zagreb,45.7984,15.98810,Europe/Berlin,HR,True
1,431,Kalisz,51.7525,18.03410,Europe/Warsaw,PL,True
2,199,Innsbruck,47.2632,11.39610,Europe/Berlin,AT,True
3,519,Siemianowice Śląskie,50.3031,19.02220,Europe/Warsaw,PL,True
4,21,Karlsruhe,49.0102,8.41827,Europe/Berlin,DE,False
...,...,...,...,...,...,...,...
262,853,Sant Just Desvern,41.3807,2.07496,Europe/Berlin,ES,True
263,856,Sant Boi de Llobregat,41.3458,2.03404,Europe/Berlin,ES,True
264,831,Wolsztyn,52.1120,16.10560,Europe/Warsaw,PL,True
265,143,Tulln,48.3269,16.05690,Europe/Berlin,AT,False


In [24]:
c = c.rename(columns={"id": "city_id", "name": "city_name"})
ts2 = new_ts.merge(c[["city_id", "city_name", "return_to_official_only"]],
                   on="city_id", how="left", validate="m:1")

In [25]:
ts2

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,...,vehicle_image,name,description,form_factor,rider_capacity,propulsion_type,max_range,battery_capacity,city_name,return_to_official_only
0,11462,7,1.669377e+09,8.253932,50.069332,8.253393,50.070504,126084480.0,NaN,0.0,...,https://static.nextbike.net/app/biketypes/type...,with Fork lock,- Comfort frame typ 21 with FVM\r\n- CFC fork ...,bicycle,1.0,NaN,NaN,NaN,Wiesbaden,False
1,11462,7,1.669897e+09,8.242628,50.071337,8.216180,50.095700,87796472.0,4775153.0,0.0,...,https://static.nextbike.net/app/biketypes/type...,with Fork lock,- Comfort frame typ 21 with FVM\r\n- CFC fork ...,bicycle,1.0,NaN,NaN,NaN,Wiesbaden,False
2,11462,7,1.670252e+09,8.216180,50.095700,8.217791,50.095744,4775153.0,NaN,0.0,...,https://static.nextbike.net/app/biketypes/type...,with Fork lock,- Comfort frame typ 21 with FVM\r\n- CFC fork ...,bicycle,1.0,NaN,NaN,NaN,Wiesbaden,False
3,11462,7,1.670349e+09,8.226448,50.076546,8.249343,50.078160,4775087.0,87796883.0,0.0,...,https://static.nextbike.net/app/biketypes/type...,with Fork lock,- Comfort frame typ 21 with FVM\r\n- CFC fork ...,bicycle,1.0,NaN,NaN,NaN,Wiesbaden,False
4,11462,7,1.671514e+09,8.249343,50.078160,8.242628,50.071337,87796883.0,87796472.0,0.0,...,https://static.nextbike.net/app/biketypes/type...,with Fork lock,- Comfort frame typ 21 with FVM\r\n- CFC fork ...,bicycle,1.0,NaN,NaN,NaN,Wiesbaden,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161234,840054,679,1.684753e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,...,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0,Penarth,True
1161235,840054,679,1.685088e+09,-3.166644,51.446057,-3.163247,51.465538,6270602.0,88102363.0,100.0,...,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0,Penarth,True
1161236,840054,679,1.686059e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,...,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0,Penarth,True
1161237,840054,679,1.686121e+09,-3.185756,51.429879,-3.163247,51.465538,86312067.0,88102363.0,87.0,...,https://static.nextbike.net/app/biketypes/type...,e-SMARTbike 2.0 with RFID,- e-eco B frame with Framelock\r\n - e-CFC co...,bicycle,1.0,1.0,60000.0,500.0,Penarth,True


In [26]:
ts2.isna().mean() * 100

bike_id                     0.000000
city_id                     0.000000
time_start                  0.000000
lon_start                   0.000000
lat_start                   0.000000
lon_end                     0.000000
lat_end                     0.000000
station_id_start            7.681967
station_id_end             10.496978
battery_start               0.000000
battery_end                 0.000000
duration                    0.000000
distance                    0.000000
id_bike                     0.000000
bike_type_id                0.000000
computer_id                 0.000000
id_bike_type                0.000000
vehicle_image               0.202026
name                        0.202026
description                48.558565
form_factor                 0.202026
rider_capacity              0.202026
propulsion_type            84.503276
max_range                  84.511457
battery_capacity           84.503276
city_name                   0.000000
return_to_official_only     0.000000
d

In [27]:
ts2["any_station_missing"] = ts2["station_id_start"].isna() | ts2["station_id_end"].isna()

base = ts2["return_to_official_only"].value_counts(normalize=True) * 100
miss = ts2.loc[ts2["any_station_missing"], "return_to_official_only"].value_counts(normalize=True) * 100
print(pd.concat([base, miss], axis=1, keys=["all trips", "station missing"]))

                         all trips  station missing
return_to_official_only                            
True                     82.706402        74.682314
False                    17.293598        25.317686


* 75% of trips with station_id that are missing require return to official station only which reject our hypothesis.
* Station ids are almost MCAR so we will delete them

In [28]:
rows = ts2["station_id_start"].isna() | ts2["station_id_end"].isna()
ts2 = ts2[~rows]
ts2.isna().mean()

bike_id                    0.000000
city_id                    0.000000
time_start                 0.000000
lon_start                  0.000000
lat_start                  0.000000
lon_end                    0.000000
lat_end                    0.000000
station_id_start           0.000000
station_id_end             0.000000
battery_start              0.000000
battery_end                0.000000
duration                   0.000000
distance                   0.000000
id_bike                    0.000000
bike_type_id               0.000000
computer_id                0.000000
id_bike_type               0.000000
vehicle_image              0.002303
name                       0.002303
description                0.452042
form_factor                0.002303
rider_capacity             0.002303
propulsion_type            0.835443
max_range                  0.835486
battery_capacity           0.835443
city_name                  0.000000
return_to_official_only    0.000000
any_station_missing        0

In [29]:

columns = ["bike_id","city_id","time_start","lon_start","lat_start","lon_end","lat_end","station_id_start","station_id_end","battery_start","battery_end","duration","distance"]
ts2 = ts2[columns]
ts2

,bike_id,city_id,time_start,lon_start,lat_start,lon_end,lat_end,station_id_start,station_id_end,battery_start,battery_end,duration,distance
1,11462,7,1.669897e+09,8.242628,50.071337,8.216180,50.095700,87796472.0,4775153.0,0.0,0.0,1920.0,3301.514764
3,11462,7,1.670349e+09,8.226448,50.076546,8.249343,50.078160,4775087.0,87796883.0,0.0,0.0,540.0,1643.609250
4,11462,7,1.671514e+09,8.249343,50.078160,8.242628,50.071337,87796883.0,87796472.0,0.0,0.0,4680.0,897.351993
5,11463,7,1.662315e+09,8.218090,50.080828,8.242628,50.071337,4728239.0,87796472.0,0.0,0.0,600.0,2044.510274
6,11463,7,1.662542e+09,8.242628,50.071337,8.252207,50.071756,87796472.0,87797085.0,0.0,0.0,1020.0,685.223644
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161234,840054,679,1.684753e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,96.0,1320.0,3061.891906
1161235,840054,679,1.685088e+09,-3.166644,51.446057,-3.163247,51.465538,6270602.0,88102363.0,100.0,97.0,780.0,2178.941052
1161236,840054,679,1.686059e+09,-3.174022,51.438833,-3.163247,51.465538,30005988.0,88102363.0,100.0,95.0,900.0,3061.891906
1161237,840054,679,1.686121e+09,-3.185756,51.429879,-3.163247,51.465538,86312067.0,88102363.0,87.0,75.0,1020.0,4260.899309


In [31]:
ts2.to_csv("../trips_small_cleaned.csv", index=False)